# Banking feedback: full dataset

Complete processing and labelling of the pinned review collection.


In [ ]:
import subprocess
import sys
import time
from pathlib import Path
notebook_started=time.monotonic()
wheel='https://github.com/vllm-project/vllm/releases/download/v0.20.0/vllm-0.20.0%2Bcu129-cp38-abi3-manylinux_2_31_x86_64.whl'
try:
    subprocess.run([sys.executable,'-m','pip','install','-q',wheel,'json-repair==0.55.0'],check=True,timeout=600)
except (subprocess.CalledProcessError,subprocess.TimeoutExpired):
    print('ACCELERATED_RUNTIME_UNAVAILABLE')
    subprocess.check_call([sys.executable,'-m','pip','install','-q','json-repair==0.55.0'])


## Source checks


In [ ]:
import runpy
import json
import time
import importlib.metadata
import pandas as pd
asset=next(Path('/kaggle/input').rglob('feedback_pipeline.py')).parent
ctx=runpy.run_path(str(asset/'feedback_pipeline.py'))
for source in asset.glob('*.py'):
    print('SOURCE_AUDIT',ctx['source_audit'](source))
subprocess.run([sys.executable,str(asset/'export_smoke_test.py')],check=True)
started=time.monotonic()


## Full-corpus labels


In [ ]:
stage_started=time.monotonic()
try:
    attempt=subprocess.run([sys.executable,str(asset/'fast_full.py')],check=False)
    if attempt.returncode:
        print('USING_TRANSFORMERS_ENGINE')
        subprocess.run([sys.executable,str(asset/'fallback_full.py')],check=True)
finally:
    print('GENERATION_STAGE_SECONDS',time.monotonic()-stage_started)


## Sentiment checks


In [ ]:
out=Path('/kaggle/working/sarah_feedback_full')
data=pd.read_parquet(out/'prepared_rows.parquet')
unique=pd.read_parquet(out/'unique_input.parquet')
records=[json.loads(line) for line in (out/'generative_records.jsonl').read_text().splitlines()]
encoder=ctx['infer_encoder'](unique,out)
labels=pd.DataFrame([{**record,'a':json.dumps(record['a'])} for record in records]).merge(pd.DataFrame(encoder),on='duplicate_group',validate='one_to_one')
print('UNIQUE_LABEL_ROWS',len(labels))


## Complete results and review worksheet


In [ ]:
results,aspects=ctx['finalize_full'](data,labels,out)
print(results.groupby(['bank','sentiment'],dropna=False).size().unstack(fill_value=0))
print(json.loads((out/'reviewer_only'/'processing_checks.json').read_text()))


## Charts


In [ ]:
ctx['make_figures'](results,aspects,out)


## Export and integrity


In [ ]:
import shutil
web=runpy.run_path(str(asset/'web_export.py'))
web['export_web'](results,out,asset)
ctx['audit_notebook'](out)
review=out/'reviewer_only'
installed={dist.metadata['Name'].lower().replace('_','-'):dist.version for dist in importlib.metadata.distributions()}
packages=['torch','transformers','vllm','json-repair','pandas','numpy','pyarrow','matplotlib']
(review/'requirements.txt').write_text('\n'.join(name+'=='+installed[name] for name in packages if name in installed)+'\n')
shutil.copy2(asset/'full_notebook_source.json',review/'submitted_notebook.json')
for name in ['fast_full.py','fallback_full.py','export_smoke_test.py','web_export.py','search_stopwords.json']:
    shutil.copy2(asset/name,review/name)
torch=ctx['torch']
hardware={'python':sys.version,'torch':torch.__version__,'cuda':torch.version.cuda,'gpu_count':torch.cuda.device_count(),'gpus':[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],'cpu_count':ctx['os'].cpu_count()}
ctx['save_json'](out/'runtime.json',hardware)
summary={'dataset_rows':len(results),'unique_nonempty_texts':len(unique),'labelled_rows':int(results['sentiment'].notna().sum()),'review_queue_rows':int(results['review_needed'].sum()),'runtime_seconds':time.monotonic()-notebook_started,'status':'complete'}
ctx['save_json'](out/'run_summary.json',summary)
archive=ctx['package_outputs'](out,'sarah_feedback_full')
print('FULL_RUN_SUMMARY',json.dumps(summary))
